# Pronóstico de ventas

## Descripción del proyecto

Este proyecto tiene el propósito de estudiar los datos de la empresa online Ice cuyo giro es la venta de videojuegos, con alcance en todo el mundo, para identificar patrones que ayuden a determinar si una próxima entrega tendría éxito o no con el fin de planificar estrategicamente campañas publicitarias.

## Importando librerías, cargando DataFrame y revisión general

In [84]:
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
from scipy import stats as st

In [85]:
df_games = pd.read_csv('./datasets/games.csv')

In [86]:
# Revisando la estructura general del DataFrame (no. de columnas, nombres de columnas, cantidad de filas, cuenta de datos no nulos y tipo de dato de las columnas)
df_games.info()

<class 'pandas.DataFrame'>
RangeIndex: 16715 entries, 0 to 16714
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16713 non-null  str    
 1   Platform         16715 non-null  str    
 2   Year_of_Release  16446 non-null  float64
 3   Genre            16713 non-null  str    
 4   NA_sales         16715 non-null  float64
 5   EU_sales         16715 non-null  float64
 6   JP_sales         16715 non-null  float64
 7   Other_sales      16715 non-null  float64
 8   Critic_Score     8137 non-null   float64
 9   User_Score       10014 non-null  str    
 10  Rating           9949 non-null   str    
dtypes: float64(6), str(5)
memory usage: 1.4 MB


En esta primera revisión con `info()`, se identificaron las siguientes observaciones sobre la estructura inicial del DataFrame:

- Se cuenta con 11 columnas donde el nombre de estas utilizan un formato (ej. `Year_of_Release`, `Critic_Score`, `User_Score`, etc.) que dificultaría la escritura para hacer referencia a ellos cuando se necesiten.
- Hay 6 columnas que cuentan con valores nulos y 5 que cuentan con valores en todas sus filas.

In [87]:
# Revisando la información de los primeros 5 registros
df_games.head()

,Name,Platform,Year_of_Release,Genre,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score,User_Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [88]:
print(f'Valores duplicados: {df_games.duplicated().sum()}\n')
print(f'Valores nulos\n{df_games.isna().sum()}')

Valores duplicados: 0

Valores nulos
Name                  2
Platform              0
Year_of_Release     269
Genre                 2
NA_sales              0
EU_sales              0
JP_sales              0
Other_sales           0
Critic_Score       8578
User_Score         6701
Rating             6766
dtype: int64


Al extraer los primeros registros puede observarse lo siguiente:
- El tipo de dato para la columna `Year_of_Release` es de tipo flotante cuando lo más indicado para este caso podría ser `object`
- Los valores encontrados en `NA_Sales`, `EU_Sales`, `JP_Sales` y `Other_sales` son de tipo flotante indicando la cantidad de ventas en millones de dólares (`USD`).
- La puntuación de la crítica (`Critic_Score`) se evalúa sobre 100 y es de tipo `float`.
- La puntuación del público (`User_Score`) se evalúa sobre 10 y es de tipo `object`.
- Aunque `Critic_Score` y `User_Score` muestren sus valores con punto decimal, ambas columnas tienen diferentes tipos de datos.
- La columna `Rating` almacena la clasificación de edad basada en la ESRB.


In [89]:
# Revisando una muestra de 10 registros aleatorios
df_games.sample(10)

,Name,Platform,Year_of_Release,Genre,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score,User_Score,Rating
3593,Yokai Sangokushi,3DS,2016.0,Action,0.00,0.00,0.56,0.00,NaN,NaN,NaN
4267,MX vs. ATV Untamed,PS3,2007.0,Racing,0.35,0.06,0.00,0.05,67.0,6.1,E
6103,Yu-Gi-Oh! Double Pack,GBA,2006.0,Role-Playing,0.20,0.08,0.00,0.01,NaN,NaN,NaN
996,Sonic the Hedgehog 3,GEN,1994.0,Platform,1.02,0.47,0.20,0.07,NaN,NaN,NaN
2246,Phoenix Wright: Ace Attorney,DS,2005.0,Adventure,0.44,0.05,0.39,0.05,81.0,9.2,T
7736,UFC Personal Trainer: The Ultimate Fitness System,PS3,2011.0,Sports,0.08,0.08,0.00,0.03,65.0,2.4,E
3409,Disney Sing It: Family Hits,Wii,2010.0,Misc,0.38,0.17,0.00,0.05,NaN,tbd,E
6844,Front Line,2600,1981.0,Action,0.22,0.01,0.00,0.00,NaN,NaN,NaN
10908,Psychonauts,PS2,2005.0,Platform,0.05,0.04,0.00,0.01,86.0,8.7,T
1402,Paper Mario,N64,2000.0,Role-Playing,0.58,0.18,0.59,0.02,NaN,NaN,NaN


Tomándo muestras aleatorias se detectó lo siguiente:
- La ausencia de ventas de un título para cualquiera de las regiones disponibles se registra con `0.0`.
- Hay valores ausentes en `Critic_Score`, `User_Score` y `Rating`.

### Resumen de las observaciones

Después de varias revisiones para la fase inicial de observación de la estructura del DataFrame se puede notar que las columnas no tienen un formato apropiado para las fases de análisis posteriores a la preparación de los datos. También, varias de las columnas necesitan convertirse a un tipo de dato más apropiado y hay tanto valores ausentes o valores posicionales temporales (ej. `tbd`) que se revisará más adelante la forma más apropiada de manejarlos en base a los requerimientos de análisis.

## Preparación los datos

### Cambiando nombre de las columnas

In [90]:
# Convirtiendo nombres de las columnas en minúscula
df_games.columns = df_games.columns.str.lower()

# Cambiando nombre de la columna `year_of_release`
df_games = df_games.rename(columns= {'year_of_release': 'release_year'})

# Verificando los nuevos nombres de columna
print(df_games.columns.tolist())

['name', 'platform', 'release_year', 'genre', 'na_sales', 'eu_sales', 'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating']


### Tratando tipos de dato y valores nulos

#### `genre`

In [91]:
df_games['genre'] = df_games['genre'].fillna('Misc')

#### `release_year`

In [92]:
# Convirtiendo tipo de dato para la columna de `release_year`
df_games['release_year'] = df_games['release_year'].astype('Int64').fillna(pd.NA)

#### `critic_score`

In [93]:
df_games['critic_score'] = (
    df_games['critic_score']
    .astype('Int64')
    .fillna(pd.NA)
)

#### `user_score`

In [94]:
df_games['user_score'] = (
    df_games['user_score']
    .replace('tbd', pd.NA)
    .astype('Float64')
    .fillna(pd.NA)
)

In [96]:
# df_games.loc[(df_games['user_score'] == 'tbd') | (df_games['user_score'].isna())]

df_games.sample(10)

,name,platform,release_year,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
13072,Monster! Bass Fishing,GBA,2003,Sports,0.04,0.01,0.00,0.00,<NA>,<NA>,NaN
9120,Pro Evolution Soccer 2015,XOne,2014,Sports,0.03,0.10,0.00,0.01,79,7.3,E
10625,Nodame Cantabile,DS,2007,Misc,0.00,0.00,0.10,0.00,<NA>,<NA>,NaN
5095,Super Formation Soccer 94,SNES,1994,Sports,0.00,0.00,0.37,0.00,<NA>,<NA>,NaN
8110,Spider-Man: Friend or Foe,PSP,2007,Action,0.16,0.00,0.00,0.01,58,4.4,E10+
11353,RTX Red Rock,PS2,2003,Shooter,0.04,0.03,0.00,0.01,49,5.0,T
14707,Superdimension Neptune vs Sega Hard Girls,PSV,2016,Role-Playing,0.00,0.02,0.00,0.01,73,8.4,T
315,Tekken 5,PS2,2005,Fighting,0.93,1.94,0.31,0.70,88,8.6,T
1246,Mega Man 2,NES,1988,Action,0.93,0.15,0.42,0.01,<NA>,<NA>,NaN
14918,Imagine: Sweet 16,DS,2010,Simulation,0.02,0.00,0.00,0.00,<NA>,<NA>,E
